# Person 4 sweep: degradation and nutrient dynamics

This notebook sweeps κ_b, d_a, φ_max, and ξ for the Ma et al. (2024) model. The treatment outputs describe antibiotic clearance, population crash depth, recovery, steady state, survival, resistant fraction, and diversity.

In [1]:
import time
import pandas as pd
from ma2024_model import SI_RANDOMIZED_RANGES, SI_INITIAL_CONDITIONS, DEFAULT_PARAMS
from sweep_engine import verify_equivalence, compute_endpoints, no_drug_reference
from grid_runner import run_treatment_grid
KAPPA_B_RANGE = SI_RANDOMIZED_RANGES['kappa_b']
PHI_MAX_RANGE = SI_RANDOMIZED_RANGES['phi_max']
XI_RANGE = SI_RANDOMIZED_RANGES['xi']
D_A_RANGE = (0.0, 0.10)  # d_a has no randomized SI range; explicit exploratory range
print('Base parameters:', DEFAULT_PARAMS)
print('Initial conditions:', SI_INITIAL_CONDITIONS)
print('Ranges:', KAPPA_B_RANGE, D_A_RANGE, PHI_MAX_RANGE, XI_RANGE)

Base parameters: Params(alpha=0.95, beta_min=0.9, c=0.7, gamma=1.38, h_a=3.0, h_i=2.0, kappa_b=0.35, d_a=0.02, d_b=1.0, xi=0.8, phi_max=1.0)
Initial conditions: {'ns0': 0.2, 'nr0': 0.2, 's0': 4.0, 'b0': 0.0}
Ranges: (0.0, 1.0) (0.0, 0.1) (0.0, 5.0) (0.0, 1.0)


## Step 1 — Verify the log-space engine

In [2]:
diffs = verify_equivalence(n_checks=30, seed=1, a0=2.15, i_dose=10.0, t_span=(0.0, 40.0))
print(diffs)
assert all(value <= 0 for value in diffs.values())
print('PASSED')

{'ns': 0.0, 'nr': 0.0, 's': 0.0, 'a': 0.0, 'b': 0.0}
PASSED


## Step 2 — Check the treatment horizon

In [3]:
ns0_ref = no_drug_reference()
corners = [(kb, da, pm, xi) for kb in KAPPA_B_RANGE for da in D_A_RANGE for pm in PHI_MAX_RANGE for xi in XI_RANGE]
worst_t_ss = 0.0
unconverged = 0
for kb, da, pm, xi in corners:
    ep = compute_endpoints(kb, da, pm, xi, a0=2.15, i_dose=10.0, ns0_no_drug=ns0_ref, t_span=(0, 300), n_points=300)
    worst_t_ss = max(worst_t_ss, ep.t_steady_state)
    unconverged += not ep.ss_converged
print(f'Worst t_SS: {worst_t_ss:.2f}; unconverged: {unconverged}/{len(corners)}')

Worst t_SS: 300.00; unconverged: 2/16


## Step 3 — Run the four-way treatment grid

In [4]:
t0 = time.time()
treatment_df = run_treatment_grid(n_per_axis=21, kappa_b_range=KAPPA_B_RANGE, d_a_range=D_A_RANGE, phi_max_range=PHI_MAX_RANGE, xi_range=XI_RANGE, a0=2.15, i_dose=10.0, t_span=(0.0, 150.0), n_points=300)
elapsed = time.time() - t0
print(f'Treatment grid: {len(treatment_df)} points in {elapsed:.1f}s')
print(f'Unconverged: {(~treatment_df.ss_converged).sum()} / {len(treatment_df)}')
treatment_df.to_csv('treatment_grid_300_tspan.csv', index=False)
print('Saved treatment_grid_300_tspan.csv')
treatment_df.describe(include='all')

Treatment grid: 194481 points in 787.3s
Unconverged: 462 / 194481
Saved treatment_grid_300_tspan.csv


,kappa_b,d_a,phi_max,xi,f_R_final,resistant_eliminated,sensitive_survives,t_recovery_S,t_steady_state,ss_converged,peak_loss_S,peak_loss_total,diversity_final,diversity_min,ns_final,nr_final
count,194481.000000,194481.000000,194481.000000,194481.000000,194481.000000,194481,194481,194481.000000,194481.000000,194481,194481.000000,194481.000000,194481.000000,194481.000000,1.944810e+05,1.944810e+05
unique,NaN,NaN,NaN,NaN,NaN,2,2,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,False,True,NaN,NaN,True,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,194133,188508,NaN,NaN,194019,NaN,NaN,NaN,NaN,NaN,NaN
mean,0.500000,0.050000,2.500000,0.500000,0.449345,NaN,NaN,9.543381,11.697160,NaN,0.238441,0.248423,0.686890,0.686890,1.975557e+00,1.623629e+00
std,0.302766,0.030277,1.513829,0.302766,0.022590,NaN,NaN,6.379348,8.935809,NaN,0.121910,0.124644,0.016221,0.016221,3.471080e-01,3.073178e-01
min,0.000000,0.000000,0.000000,0.000000,0.084114,NaN,NaN,0.000000,7.525084,NaN,0.110105,0.115324,0.288704,0.288704,6.833792e-15,1.673925e-15
25%,0.250000,0.025000,1.250000,0.250000,0.447969,NaN,NaN,7.023411,8.528428,NaN,0.158823,0.166262,0.687723,0.687723,1.815673e+00,1.492596e+00
50%,0.500000,0.050000,2.500000,0.500000,0.454420,NaN,NaN,8.026756,10.033445,NaN,0.203985,0.213345,0.688986,0.688986,2.030141e+00,1.688327e+00
75%,0.750000,0.075000,3.750000,0.750000,0.457916,NaN,NaN,10.033445,12.040134,NaN,0.279071,0.291235,0.689601,0.689601,2.227946e+00,1.844248e+00


In [ ]:
t_300 = 